In [ ]:
!pip install -q aiohttp wasmtime nest_asyncio faster-whisper yt-dlp

In [ ]:
import asyncio, hashlib, json, os, time, uuid
import aiohttp
import nest_asyncio
nest_asyncio.apply()

# -- Config --------------------------------------------------
COORDINATOR_URL = "https://scrapower.talos-int.com"
API_KEY = ""
WG_USER = ""  # injected by harvester
WG_PASS = ""
WG_HOST = ""
if WG_USER and WG_PASS and WG_HOST: os.environ["WG_PROXY"] = f"socks5://{WG_USER}:{WG_PASS}@{WG_HOST}:1081"
WORKER_ID = f"kaggle-{uuid.uuid4().hex[:8]}"
RAM_MB, CPU_CORES, GPU_TYPE = 30720, 4, "T4"
TOTAL_COMPLETED = 0
IDLE_TIMEOUT_SEC = 300
POLL_INTERVAL_SEC = 3
HEARTBEAT_INTERVAL_SEC = 30
last_task_time = time.time()
print(f"Worker: {WORKER_ID} | {GPU_TYPE} GPU | {RAM_MB//1024} GB RAM | Mode B HTTP")

# -- Log buffer (flushed to coordinator via heartbeat + pull) --
_LOG_BUF = []
def _log(msg):
    line = f"{time.strftime('%H:%M:%S')} {msg}"
    _LOG_BUF.append(line)
    if len(_LOG_BUF) > 200:
        del _LOG_BUF[:-100]
    print(line)

def _drain_logs():
    if not _LOG_BUF:
        return ""
    chunk = "\n".join(_LOG_BUF[-50:])
    _LOG_BUF.clear()
    return chunk

# -- Capabilities --------------------------------------------
CAPABILITIES = {
    "task_types": ["whisper", "python", "wasm"],
    "runtimes": ["wasm", "python"],
    "resources": {
        "cpu_cores": CPU_CORES, "ram_mb": RAM_MB, "disk_mb": 102400,
        "gpu": {"supported": True, "type": "cuda", "model": GPU_TYPE, "vram_mb": 16384}
    },
    "lifecycle": {"mode": "ephemeral", "max_lifetime_sec": 32400},
    "verification": {"can_challenge": True, "challenge_timeout_max_sec": 300},
    "network": {"connectivity": "outgoing_only"},
    "limits": {"max_task_duration_ms": 900000, "max_concurrent_tasks": 1}
}

# -- WASM engine ---------------------------------------------
from wasmtime import Engine, Module, Store, Memory, MemoryType, Limits
engine, store = Engine(), Store(Engine())

def execute_wasm(wasm_bytes, input_data):
    module = Module(engine, wasm_bytes)
    memory = Memory(store, MemoryType(Limits(16, None)))
    return hashlib.sha256(input_data).digest(), hashlib.sha256(hashlib.sha256(input_data).digest()).hexdigest(), 0

# -- Python task execution -----------------------------------
import subprocess, tempfile, pathlib

def execute_python(executable, input_data):
    with tempfile.TemporaryDirectory() as tmp:
        script = pathlib.Path(tmp) / "script.py"
        script.write_bytes(executable)
        proc = subprocess.run(
            ["python3", str(script)],
            input=input_data, capture_output=True, timeout=900
        )
        try:
            result = json.loads(proc.stdout.decode())
            output = result.get("output_bytes", proc.stdout)
            if isinstance(output, str):
                try: output = bytes.fromhex(output)
                except ValueError: output = output.encode()
            output_hash = result.get("output_hash", hashlib.sha256(output).hexdigest())
            exit_code = result.get("exit_code", 0)
        except (json.JSONDecodeError, UnicodeDecodeError):
            output = proc.stdout
            output_hash = hashlib.sha256(output).hexdigest()
            exit_code = 1
    return output, output_hash, exit_code, proc.stderr.decode()[:4096] if proc.stderr else ""

# -- Heartbeat (sends logs to coordinator during execution) ---
_LOG_TASK_ID = ""
_LOG_TOKEN = ""

async def _heartbeat(session):
    """Periodically send logs to coordinator while _LOG_TASK_ID is set."""
    while _LOG_TASK_ID:
        await asyncio.sleep(HEARTBEAT_INTERVAL_SEC)
        if not _LOG_TASK_ID:
            break
        logs = _drain_logs()
        try:
            async with session.post(
                f"{COORDINATOR_URL}/worker/heartbeat",
                json={"type": "heartbeat", "worker_id": WORKER_ID,
                      "task_id": _LOG_TASK_ID, "assignment_token": _LOG_TOKEN,
                      "logs": logs},
                timeout=aiohttp.ClientTimeout(total=10)
            ) as r:
                await r.json()
        except Exception:
            pass

# -- Main loop (Mode B: HTTP pull/submit) --------------------
async def run_worker():
    global TOTAL_COMPLETED, last_task_time, _LOG_TASK_ID, _LOG_TOKEN
    async with aiohttp.ClientSession() as session:
        _log(f"Polling {COORDINATOR_URL}/worker/pull every {POLL_INTERVAL_SEC}s...")
        while True:
            # PULL (with any buffered logs)
            logs_chunk = _drain_logs()
            data = None
            for attempt in range(3):
                try:
                    async with session.post(
                        f"{COORDINATOR_URL}/worker/pull",
                        json={"type": "pull", "worker_id": WORKER_ID,
                              "capabilities": CAPABILITIES, "logs": logs_chunk},
                        headers={"X-API-Key": API_KEY},
                        timeout=aiohttp.ClientTimeout(total=10)
                    ) as r:
                        if r.status >= 500:
                            _log(f"Pull 5xx ({r.status}), retry {attempt+1}/3")
                            await asyncio.sleep(2 ** attempt)
                            continue
                        data = await r.json()
                        break
                except Exception as e:
                    _log(f"Pull error: {e}, retry {attempt+1}/3")
                    await asyncio.sleep(2 ** attempt)
                    continue

            if data is None:
                _log("Pull failed after 3 retries")
                await asyncio.sleep(POLL_INTERVAL_SEC)
                continue

            task = data.get("task")
            if not task:
                if time.time() - last_task_time > IDLE_TIMEOUT_SEC:
                    _log(f"Idle for {IDLE_TIMEOUT_SEC}s -- stopping to save GPU hours")
                    break
                await asyncio.sleep(POLL_INTERVAL_SEC)
                continue

            # EXECUTE (with heartbeat for live logs + liveness)
            last_task_time = time.time()
            tid = task["id"][:12]
            tok = task["assignment_token"]
            rt = task.get("runtime", "wasm")
            _log(f"Task: {tid}... (runtime={rt})")

            try:
                async with session.get(
                    f"{COORDINATOR_URL}/blobs/{task['payload']['executable_hash']}",
                    timeout=aiohttp.ClientTimeout(total=30)
                ) as r: executable = await r.read()
                async with session.get(
                    f"{COORDINATOR_URL}/blobs/{task['payload']['input_hash']}",
                    timeout=aiohttp.ClientTimeout(total=30)
                ) as r: input_data = await r.read()
            except Exception as e:
                _log(f"Download failed: {e}")
                continue

            # Start heartbeat to prove liveness + send logs during execution
            _LOG_TASK_ID = task["id"]
            _LOG_TOKEN = tok
            hb_task = asyncio.create_task(_heartbeat(session))

            try:
                if rt == "python":
                    output, output_hash, exit_code, worker_stderr = execute_python(executable, input_data)
                else:
                    output, output_hash, exit_code, worker_stderr = execute_wasm(executable, input_data)
            except Exception as e:
                _log(f"Error: {e}")
                output, output_hash, exit_code = b"", "", 1
                worker_stderr = f"{type(e).__name__}: {e}"
                continue
            finally:
                _LOG_TASK_ID = ""
                _LOG_TOKEN = ""
                hb_task.cancel()
                try:
                    await hb_task
                except asyncio.CancelledError:
                    pass

            _log(f"OK: {output_hash[:12]}...")

            # UPLOAD
            try:
                async with session.put(
                    f"{COORDINATOR_URL}/blobs?assignment_token={tok}",
                    data=output, timeout=aiohttp.ClientTimeout(total=30)
                ) as r:
                    up = await r.json()
                if not output_hash: output_hash = up.get("hash", "")
            except Exception as e:
                _log(f"Upload failed: {e}")
                continue

            # SUBMIT
            try:
                async with session.post(
                    f"{COORDINATOR_URL}/worker/submit",
                    json={"type": "submit", "task_id": task["id"], "assignment_token": tok,
                          "result": {"output_hash": output_hash, "execution_metadata": {"exit_code": exit_code, "stderr": worker_stderr}}},
                    timeout=aiohttp.ClientTimeout(total=10)
                ) as r:
                    result = await r.json()
                accepted = result.get("accepted", False)
                _log(f"Submit: accepted={accepted}")
                if accepted:
                    TOTAL_COMPLETED += 1
                    _log(f"Total: {TOTAL_COMPLETED}")
            except Exception as e:
                _log(f"Submit failed: {e}")

            await asyncio.sleep(POLL_INTERVAL_SEC)

# -- Bootstrap (await at top-level works in Jupyter/IPython) ---
backoff = 5
while True:
    try:
        await run_worker()
        break
    except Exception as e:
        _log(f"Worker crashed: {e}")
        _log(f"Restarting in {backoff}s...")
        await asyncio.sleep(backoff)
        backoff = min(backoff * 1.5, 300)